## Neighbourhood simulation
question: Where does a sound-driven signal actually go in the real wiring, when the path is not picked in advance?

---

### Init

In [ ]:
import pandas as pd
from pathlib import Path
import pyarrow.dataset as ds
import pyarrow.compute as pc
import duckdb
import networkx as nx

root = Path.cwd().parent
# root = Path(__file__).resolve().parent.parent

neurons_file = root / 'data' / 'body-annotations-male-cns-v1.0-minconf-0.5.feather'
connections_file = root / 'data' / 'connectome-weights-male-cns-v1.0-minconf-0.5.feather'

df = pd.read_feather(neurons_file) # directly to pandas dataframe
traced = df[df['status'] == 'Traced']
id_to_superclass = traced.set_index('bodyId')['superclass'].to_dict()

# lazy handle on the big file, nothing is loaded until a filtered read asks for it
dataset = ds.dataset(connections_file, format="feather")


# common functions and lookups copied from the earlier sections of 05

jo_test_id = 42619  # JO-B1_a, the neuron hand-traced in 04

# get outgoing connections of specified neurons
def query_edges(neuron_ids, min_weight=3):
    if not isinstance(neuron_ids, (list, set, tuple)):
        neuron_ids = [neuron_ids]
    expr = pc.field("body_pre").isin(neuron_ids) & (pc.field("weight") >= min_weight)
    return dataset.to_table(filter=expr).to_pandas()

def grow_neighbourhood(start_ids, n_hops, min_weight=3):
    if not isinstance(start_ids, (list, set, tuple)):
        start_ids = [start_ids]

    seen = set(start_ids)       # every neuron reached so far, starting ones included
    to_check = set(start_ids)   # neurons whose outgoing connections we look up this hop
    all_edges = []

    for hop in range(1, n_hops + 1):
        hop_edges = query_edges(to_check, min_weight=min_weight)
        all_edges.append(hop_edges)

        to_check = set(hop_edges['body_post']) - seen   # targets not seen before
        # seen |= to_check
        seen.update(to_check)

        # prints each hop statistics
        print(f"hop {hop}: {len(hop_edges)} edges, {len(to_check)} new neurons, {len(seen)} total")
        if not to_check:
            break

    return pd.concat(all_edges, ignore_index=True), seen

#### LIF Model

**goal (LIF model):** make a simple model of charge moving through the wiring, and have it hand back what happened (which neurons fired, and when) so "where did the signal go, and when?" can be answered for thousands of neurons.

the old LIF loop in `03` printed everything each tick, which was fine for 6 neurons. but for thousands of neurons that won't be viable. so the function has to hand back something digestible.

What should it return so the following can be answered "where did the signal go, and when?"
- print when the external input is added
- print when vnc_motor fires on which tick, along with the somaneuromere and instance
- return the full list of firing events

In [10]:
motor_info = (traced[traced['superclass'] == 'vnc_motor']
              .set_index('bodyId')[['somaNeuromere', 'instance']]
              .to_dict('index'))

def run_lif(edges, external_input, pulse_ticks, threshold, decay_factor, time_steps, verbose=True):
    # who each neuron sends charge to: {neuron: [(target, weight), ...]}
    outgoing = {}
    for pre, post, weight in edges[['body_pre', 'body_post', 'weight']].itertuples(index=False):
        outgoing.setdefault(pre, []).append((post, weight))

    charge = {}   # only neurons currently holding charge; a missing neuron means 0
    events = []   # (tick, neuron) for every firing

    for t in range(time_steps):
        if t in pulse_ticks:
            if verbose:
                print(f"tick {t}: external input added to {len(external_input)} neuron(s)")
            for nid, amount in external_input.items():
                charge[nid] = charge.get(nid, 0) + amount

        # who fires this tick, judged on charge before anything resets
        fired = [nid for nid, c in charge.items() if c >= threshold]

        # leak for everyone
        for nid in charge:
            charge[nid] *= decay_factor

        # firing neurons pass their weights on, then reset
        for nid in fired:
            events.append((t, nid))
            for post, weight in outgoing.get(nid, []):
                charge[post] = charge.get(post, 0) + weight
        for nid in fired:
            charge.pop(nid, None)
            if verbose and nid in motor_info:
                info = motor_info[nid]
                print(f"tick {t}: vnc_motor {nid} fired ({info['somaNeuromere']}, {info['instance']})")

    return events # returns a set of tick, id

##### Modifying the Threshold

> every edge in this neighborhood has weight 10 or more, because of the cutoff
>
> with a threshold of 10 or lower, any neuron that fires pushes every neuron it connects to over the "line". the signal spreads to everything reachable, and "where did it go" has the trivial answer: "everywhere"
>
> with a threshold above most weights, a neuron needs charge from several sources to fire, which makes the result selective

try to decide with real numbers:

In [11]:
edges_10, seen_10 = grow_neighbourhood(jo_test_id, n_hops=4, min_weight=10)
print()
print(edges_10['weight'].describe(percentiles=[.5, .75, .9]))

hop 1: 15 edges, 15 new neurons, 16 total
hop 2: 815 edges, 338 new neurons, 354 total
hop 3: 20718 edges, 6495 new neurons, 6849 total
hop 4: 350953 edges, 49504 new neurons, 56353 total

count    372501.000000
mean         29.274595
std          35.555511
min          10.000000
50%          18.000000
75%          31.000000
90%          57.000000
max        1352.000000
Name: weight, dtype: float64


> recognize that at a threshold of 18, half of all connections can fire their target alone, and so on for higher percentiles
>
> this is the case for a SINGLE** connection. more about this later

remember the starting neuron, the weights were:

```python
     body_pre  body_post  weight
0       42619      15233      46
1       42619     525475      45
2       42619      13014      42
3       42619      11378      42
4       42619      12066      39
```

meaning, above 46 and nothing will fire. so try to run the same simulation at different thresholds to see any changes:

In [12]:
for threshold in [10, 20, 30, 40, 50]:
    events = run_lif(edges_10, {jo_test_id: 100}, {0}, threshold, 0.8, 20, verbose=False)
    fired_ids = {nid for t, nid in events}
    motor_events = [(t, nid) for t, nid in events if nid in motor_info]
    first_motor = min((t for t, nid in motor_events), default=None)
    print(f"threshold {threshold}: {len(fired_ids)} neurons fired, "
          f"{len({nid for t, nid in motor_events})} motor neurons, first motor tick {first_motor}")

threshold 10: 56353 neurons fired, 601 motor neurons, first motor tick 3
threshold 20: 56015 neurons fired, 601 motor neurons, first motor tick 3
threshold 30: 47450 neurons fired, 585 motor neurons, first motor tick 3
threshold 40: 40696 neurons fired, 573 motor neurons, first motor tick 3
threshold 50: 1 neurons fired, 0 motor neurons, first motor tick None


- thresholds 10 to 40, the signal reaches most of the neighborhood
- threshold 50 fires, the one injected into

the next step could be to look between 40 and 50 since that is when most change happens, or start from all 138 hearing neurons instead of just one

also what should be checked is if the tick amount increases from 3, meaning more hops would've been made, so try to narrow done on that 40 to 50 range. the strongest edges from the starting neuron is 46, 45, 42, 42 and 39 so that is why the sharp drop from 40 to 50 occurs.

this time, the segment along with how many times a neuron has fired in that segment will also be added, that way it's easy to distinguish if the motor neurons are leg neurons are being fired:

In [13]:
from collections import Counter

for threshold in range(41, 48):
    events = run_lif(edges_10, {jo_test_id: 100}, {0}, threshold, 0.8, 20, verbose=False)
    fired_ids = {nid for t, nid in events}
    motor_ids = {nid for nid in fired_ids if nid in motor_info}
    first_motor = min((t for t, nid in events if nid in motor_info), default=None)
    segments = Counter(motor_info[nid]['somaNeuromere'] for nid in motor_ids)
    print(f"threshold {threshold}: {len(fired_ids)} fired, {len(motor_ids)} motor, "
          f"first motor tick {first_motor}, segments {dict(segments)}")

threshold 41: 40238 fired, 571 motor, first motor tick 3, segments {'T1': 136, 'T2': 159, 'A1': 46, 'T3': 116, 'A7': 4, 'A3': 20, 'A9': 14, 'A5': 14, 'A8': 12, 'A6': 8, 'A2': 21, 'A4': 20, 'A10': 1}
threshold 42: 40200 fired, 571 motor, first motor tick 3, segments {'T1': 136, 'T2': 159, 'A1': 46, 'T3': 116, 'A7': 4, 'A3': 20, 'A9': 14, 'A5': 14, 'A8': 12, 'A6': 8, 'A2': 21, 'A4': 20, 'A10': 1}
threshold 43: 39399 fired, 571 motor, first motor tick 4, segments {'T1': 136, 'T2': 159, 'A1': 46, 'T3': 116, 'A7': 4, 'A3': 20, 'A9': 14, 'A5': 14, 'A8': 12, 'A6': 8, 'A2': 21, 'A4': 20, 'A10': 1}
threshold 44: 39152 fired, 571 motor, first motor tick 4, segments {'T1': 136, 'T2': 159, 'A1': 46, 'T3': 116, 'A7': 4, 'A3': 20, 'A9': 14, 'A5': 14, 'A8': 12, 'A6': 8, 'A2': 21, 'A4': 20, 'A10': 1}
threshold 45: 39110 fired, 571 motor, first motor tick 4, segments {'T1': 136, 'T2': 159, 'A1': 46, 'T3': 116, 'A7': 4, 'A3': 20, 'A9': 14, 'A5': 14, 'A8': 12, 'A6': 8, 'A2': 21, 'A4': 20, 'A10': 1}
thres

> as expected, the cliff at threshold 47 exists because the strongest edge at the starting neuron is 46.
>
> something new: the "delay" isn’t fixed. the first motor tick moving from 3 to 4 and then 5 means the fast routes are being cut as the threshold rises, and the signal is taking longer ones. in `03` the delay was a fixed 5 ticks because the path was chosen. here the delay depends on which routes are open, and the threshold decides that.
> - how fast the signal arrives isn’t a property of the fly alone, it also depends on how easily neurons fire
>
> the motor neurons that fire are in every segment, including the abdomen. the signal isn’t singling out the legs. don't read that as "the fly moves its abdomen when it hears a sound", but weights have no sign, so nothing can suppress anything.

just to restate how things work again, at threshold 47, the starting neuron receives 100 at tick 0. That’s above 47, so it fires, and it counts as the 1 in “1 neurons fired”. when the starting neuron fires, it sends each of its targets that edge’s weight, so a target connected at weight 46 gets 46 added to its charge. but 46 is below 47, so that target doesn’t fire. Its charge just sits there and shrinks by the leak each tick.

when mentioning "weights" what is meant is that, a synapse can do one of two things to the neuron on the receiving end:
- excite it: push it toward firing, like adding water to the bucket
- inhibit it: push it away from firing, like draining the bucket

which one it is depends on the chemical the sending neuron releases, and the weight doesn’t record that. so “46” says there are 46 synapses, but not whether they help or hold back. the current model treats every connection as "excitatory" meaning charge is always added but never removed. real circuits use inhibition to control where a signal goes: inhibitory neurons along the way can silence the abdominal side while the leg side gets through.

> continuing on with the output analysis, the counts are mostly t1 t2 and t3. they are all > 100 while abdomen are < 100. it shouldn't be said right away that legs fire the most, but a possibility could be that there are much more leg neurons compared to abdominal neurons which inflates that number.

so knowing that possibility, the next step is to compare the fraction of each segment’s motor neurons that fired rather than totals by itself:

In [14]:
motor_info = (traced[traced['superclass'] == 'vnc_motor']
              .set_index('bodyId')[['somaNeuromere', 'instance']]
              .fillna({'somaNeuromere': 'unknown'}) # added missing values
              .to_dict('index'))

from collections import Counter

totals = Counter(info['somaNeuromere'] for info in motor_info.values())
print(totals)

for threshold in range(41, 48):
    events = run_lif(edges_10, {jo_test_id: 100}, {0}, threshold, 0.8, 20, verbose=False)
    fired_ids = {nid for t, nid in events}
    motor_ids = {nid for nid in fired_ids if nid in motor_info}
    first_motor = min((t for t, nid in events if nid in motor_info), default=None)
    segments = Counter(motor_info[nid]['somaNeuromere'] for nid in motor_ids)
    print(f"threshold {threshold}: {len(fired_ids)} fired, {len(motor_ids)} motor, "
          f"first motor tick {first_motor}, segments {dict(segments)}")

    # new
    print({seg: f"{segments[seg]}/{totals[seg]}" for seg in sorted(totals)})
    print()

Counter({'T2': 175, 'T1': 173, 'T3': 152, 'A1': 56, 'A2': 28, 'A4': 22, 'A3': 22, 'A9': 21, 'A8': 18, 'A5': 16, 'A10': 10, 'A6': 8, 'A7': 6, 'unknown': 1})
threshold 41: 40238 fired, 571 motor, first motor tick 3, segments {'T1': 136, 'T2': 159, 'A1': 46, 'T3': 116, 'A7': 4, 'A3': 20, 'A9': 14, 'A5': 14, 'A8': 12, 'A6': 8, 'A2': 21, 'A4': 20, 'A10': 1}
{'A1': '46/56', 'A10': '1/10', 'A2': '21/28', 'A3': '20/22', 'A4': '20/22', 'A5': '14/16', 'A6': '8/8', 'A7': '4/6', 'A8': '12/18', 'A9': '14/21', 'T1': '136/173', 'T2': '159/175', 'T3': '116/152', 'unknown': '0/1'}

threshold 42: 40200 fired, 571 motor, first motor tick 3, segments {'T1': 136, 'T2': 159, 'A1': 46, 'T3': 116, 'A7': 4, 'A3': 20, 'A9': 14, 'A5': 14, 'A8': 12, 'A6': 8, 'A2': 21, 'A4': 20, 'A10': 1}
{'A1': '46/56', 'A10': '1/10', 'A2': '21/28', 'A3': '20/22', 'A4': '20/22', 'A5': '14/16', 'A6': '8/8', 'A7': '4/6', 'A8': '12/18', 'A9': '14/21', 'T1': '136/173', 'T2': '159/175', 'T3': '116/152', 'unknown': '0/1'}

threshold 43

> per segment, the fractions are about equal. the raw counts that looked leg-biased came from the legs simply having more motor neurons. the abdomen isn’t being avoided. at the current model, the signal reaches nearly every segment, so it isn’t selective.

an option would be to run a control (try firing from a non-hearing neuron) but that may also fire all the neurons just like this one. so the most promising next step is to start from all 138 hearing neurons. with many sources feeding the same neurons, the threshold can be raised past 47 so that a neuron only fires if several inputs converge on it, and that’s what should make the result selective.

right now hold off from changing signs for inhibition, and focus on drawing results from the base data. here is starting from all 138 hearing neurons:

In [15]:
# regrow neighbourhood again
jo_neurons = traced[traced['type'].str.contains('JO', na=False)]
jo_auditory = jo_neurons[jo_neurons['type'].str.contains('JO-A|JO-B', na=False, regex=True)]
jo_ids = set(jo_auditory['bodyId'])
print(len(jo_ids))   # should be 138

edges_jo, seen_jo = grow_neighbourhood(jo_ids, n_hops=4, min_weight=10)

138
hop 1: 1159 edges, 335 new neurons, 473 total
hop 2: 20534 edges, 7413 new neurons, 7886 total
hop 3: 345251 edges, 57210 new neurons, 65096 total
hop 4: 1412733 edges, 89566 new neurons, 154662 total


> compare this to the results of the one neuron 4 hop test:
>
> ```python
> hop 1: 60 edges, 60 new neurons, 61 total
> hop 2: 7081 edges, 4400 new neurons, 4461 total
> hop 3: 539731 edges, 342343 new neurons, 346804 total
> hop 4: 8985411 edges, 5219030 new neurons, 5565834 total
> ```
>
> hop 4 is crazy, it almost reached the total neurons of that dataset, which is ~166k (traced) neurons.

checking how many of those neurons are traced:

In [16]:
print(sum(nid in id_to_superclass for nid in seen_jo), "of", len(seen_jo), "are traced neurons")

136058 of 154662 are traced neurons


check the percentiles like before:

In [17]:
# print(edges_jo['weight'].describe(percentiles=[.5, .75, .9]))

# to print in non-scientific notation
with pd.option_context('display.float_format', '{:,.1f}'.format):
    print(edges_jo['weight'].describe(percentiles=[.5, .75, .9]))

count   1,779,677.0
mean           26.5
std            31.4
min            10.0
50%            17.0
75%            28.0
90%            50.0
max         2,591.0
Name: weight, dtype: float64


> with a mean closer to the min, it says that most numbers are lower. also a cutoff of 17 would mean cutting 50% of entries.

try seeing what the numbers would look like for a higher cutoff:

In [18]:
edges_20, seen_20 = grow_neighbourhood(jo_ids, n_hops=4, min_weight=20)
print(edges_20['weight'].describe(percentiles=[.5, .75, .9]))

hop 1: 427 edges, 160 new neurons, 298 total
hop 2: 5536 edges, 2444 new neurons, 2742 total
hop 3: 68221 edges, 16702 new neurons, 19444 total
hop 4: 350435 edges, 37032 new neurons, 56476 total
count    424619.000000
mean         48.369147
std          46.927787
min          20.000000
50%          33.000000
75%          53.000000
90%          89.000000
max        2591.000000
Name: weight, dtype: float64


trying the 10 cutoff, with 20 threshold:

In [19]:
order = ['T1', 'T2', 'T3'] + [f'A{i}' for i in range(1, 11)] + ['unknown']

def summarize(events):
    fired_ids = {nid for t, nid in events}
    motor_ids = {nid for nid in fired_ids if nid in motor_info}
    first_motor = min((t for t, nid in events if nid in motor_info), default=None)
    segments = Counter(motor_info[nid]['somaNeuromere'] for nid in motor_ids)
    print(f"{len(fired_ids)} fired, {len(motor_ids)} motor, first motor tick {first_motor}")
    for seg in order:
        print(f"{seg}: {segments[seg]}/{totals[seg]} ({segments[seg] / totals[seg]:.0%})")

# edges_jo is the 10 cutoff, and see the 20 arg for the threshold. there's also 20 ticks
events = run_lif(edges_jo, {nid: 100 for nid in jo_ids}, {0}, 20, 0.8, 20, verbose=False)
summarize(events)

153428 fired, 672 motor, first motor tick 2
T1: 158/173 (91%)
T2: 170/175 (97%)
T3: 140/152 (92%)
A1: 56/56 (100%)
A2: 28/28 (100%)
A3: 22/22 (100%)
A4: 22/22 (100%)
A5: 16/16 (100%)
A6: 8/8 (100%)
A7: 6/6 (100%)
A8: 18/18 (100%)
A9: 21/21 (100%)
A10: 6/10 (60%)
unknown: 1/1 (100%)


> see here it's different for the full 138 neurons firing case. before for a single connection, it was easy to look at the median number and say 50% of the connections would drop because the threshold was put at the 50th percentile. however in a multiple neuron connection, that threshold can be met from several inputs added together (compounding effect). that is why despite putting a weight above the 50th percentile (17), a lot of the neurons still fire.

to actually see the compounding effect, this can be done:

In [20]:
print(edges_jo.groupby('body_post')['weight'].sum().describe(percentiles=[.5, .75, .9]))

count    154594.000000
mean        304.759337
std         999.386324
min          10.000000
50%          64.000000
75%         180.000000
90%         656.000000
max       55067.000000
Name: weight, dtype: float64


> total incoming weight per neuron has a median of 64, against 17 for a single edge. the 75th percentile is 180, the 90th is 656, and the maximum is 55,067. so a typical neuron receives about four times the weight of a typical edge, and a threshold of 20 is easily reached by a neuron with a few firing inputs.
>
> also notice that the line is a slightly different version of `print(edges_20['weight'].describe(percentiles=[.5, .75, .9]))`
>
> the distribution to look at is total incoming weight per neuron, not per edge. because the simulation fires a neuron when its summed charge reaches the threshold, the total incoming weight is the most charge a neuron can ever receive in one tick. that makes it the ceiling on how easy that neuron is to fire.


moving on, the result previously is considered, the baseline, and it’s useful as one that answers: "at a low threshold, everything fires". so the question now is where does everything stop firing?

as threshold goes up its expected that less neurons fire, and to call the result selective is when an area (whether it be the abdomen or the leg muscles) has more % of firing than another area. next, need try to scan thresholds between 20 -> 50:

In [21]:
leg_segments = ['T1', 'T2', 'T3']
abdomen_segments = [f'A{i}' for i in range(1, 11)]

def group_fraction(segments, group):
    return sum(segments[s] for s in group) / sum(totals[s] for s in group)

sources = {nid: 100 for nid in jo_ids}

for threshold in range(20, 55, 5):
    events = run_lif(edges_jo, sources, {0}, threshold, 0.8, 20, verbose=False)
    fired_ids = {nid for t, nid in events}
    motor_ids = {nid for nid in fired_ids if nid in motor_info}
    first_motor = min((t for t, nid in events if nid in motor_info), default=None)
    segments = Counter(motor_info[nid]['somaNeuromere'] for nid in motor_ids)
    print(f"threshold {threshold}: {len(fired_ids)} fired, "
          f"legs {group_fraction(segments, leg_segments):.0%}, "
          f"abdomen {group_fraction(segments, abdomen_segments):.0%}, "
          f"first motor tick {first_motor}")

threshold 20: 153428 fired, legs 94%, abdomen 98%, first motor tick 2
threshold 25: 151382 fired, legs 94%, abdomen 98%, first motor tick 2
threshold 30: 132949 fired, legs 93%, abdomen 98%, first motor tick 2
threshold 35: 126373 fired, legs 92%, abdomen 98%, first motor tick 2
threshold 40: 120221 fired, legs 91%, abdomen 98%, first motor tick 2
threshold 45: 115681 fired, legs 91%, abdomen 98%, first motor tick 2
threshold 50: 110974 fired, legs 91%, abdomen 98%, first motor tick 2


> so it was not realized until later, but 50 is the 90th percentile of single-edge weights, but that is the wrong yardstick here, because a neuron’s charge is the sum of many inputs

because 20 -> 50 still had many neurons firing, scan the thresholds past the initial range. this says how much higher to go before the signal stopped, and the 95%, 99% and 99.9% values (73, 154, 345) gives landmarks for it:

In [22]:
print(edges_jo['weight'].quantile([.95, .99, .999]))

0.950     73.0
0.990    154.0
0.999    345.0
Name: weight, dtype: float64


using the new thresholds from above, update the loop:

In [23]:
sources = {nid: 10_000 for nid in jo_ids}

for threshold in [60, 80, 100, 150, 200, 300]:
    events = run_lif(edges_jo, sources, {0}, threshold, 0.8, 20, verbose=False)
    fired_ids = {nid for t, nid in events}
    motor_ids = {nid for nid in fired_ids if nid in motor_info}
    first_motor = min((t for t, nid in events if nid in motor_info), default=None)
    segments = Counter(motor_info[nid]['somaNeuromere'] for nid in motor_ids)
    print(f"threshold {threshold}: {len(fired_ids)} fired, "
          f"legs {group_fraction(segments, leg_segments):.0%}, "
          f"abdomen {group_fraction(segments, abdomen_segments):.0%}, "
          f"first motor tick {first_motor}")

threshold 60: 101114 fired, legs 90%, abdomen 98%, first motor tick 2
threshold 80: 82683 fired, legs 88%, abdomen 98%, first motor tick 3
threshold 100: 69841 fired, legs 86%, abdomen 98%, first motor tick 3
threshold 150: 56751 fired, legs 83%, abdomen 97%, first motor tick 3
threshold 200: 49433 fired, legs 80%, abdomen 96%, first motor tick 3
threshold 300: 34158 fired, legs 76%, abdomen 91%, first motor tick 5


> the abdomen consistently fires more than the legs, and the gap grows. abdominal motor neurons are easier to excite.
>
> the opposite of what was observed before is happening (the signal prefers the legs)

two things to check:
- try to compare how much total incoming weight leg and abdominal motor neurons receive to see if they really are easier to excite
- extend the scan upward until the signal dies and see how big the gap is between legs and abdomen firing

first, try extending the scan:

In [24]:
for threshold in [310, 320, 330, 340, 350, 360, 361, 362, 363, 364, 365, 370, 380, 390, 400, 600, 1000, 1500, 2500]:
    events = run_lif(edges_jo, sources, {0}, threshold, 0.8, 20, verbose=False)
    fired_ids = {nid for t, nid in events}
    motor_ids = {nid for nid in fired_ids if nid in motor_info}
    first_motor = min((t for t, nid in events if nid in motor_info), default=None)
    segments = Counter(motor_info[nid]['somaNeuromere'] for nid in motor_ids)
    print(f"threshold {threshold}: {len(fired_ids)} fired, "
          f"legs {group_fraction(segments, leg_segments):.0%}, "
          f"abdomen {group_fraction(segments, abdomen_segments):.0%}, "
          f"first motor tick {first_motor}")

threshold 310: 32841 fired, legs 75%, abdomen 89%, first motor tick 5
threshold 320: 29694 fired, legs 75%, abdomen 88%, first motor tick 6
threshold 330: 27019 fired, legs 74%, abdomen 88%, first motor tick 8
threshold 340: 23098 fired, legs 72%, abdomen 86%, first motor tick 8
threshold 350: 17021 fired, legs 69%, abdomen 75%, first motor tick 8
threshold 360: 7914 fired, legs 61%, abdomen 29%, first motor tick 9
threshold 361: 8091 fired, legs 61%, abdomen 27%, first motor tick 9
threshold 362: 163 fired, legs 0%, abdomen 0%, first motor tick None
threshold 363: 156 fired, legs 0%, abdomen 0%, first motor tick None
threshold 364: 156 fired, legs 0%, abdomen 0%, first motor tick None
threshold 365: 156 fired, legs 0%, abdomen 0%, first motor tick None
threshold 370: 155 fired, legs 0%, abdomen 0%, first motor tick None
threshold 380: 153 fired, legs 0%, abdomen 0%, first motor tick None
threshold 390: 152 fired, legs 0%, abdomen 0%, first motor tick None
threshold 400: 152 fired, leg

> finally, the pattern finally flips and becomes selective. mainly legs are firing at threshold 360-361.
>
> threshold 362+ is a real cliff and so some signal depends on a narrow gate